# Prédiction du prix du Bitcoin avec un réseau LSTM (Keras)

Notebook de l'article [Prédiction du prix des crypto monnaies avec du Deep Learning](https://moncoachdata.com/blog/prediction-du-prix-des-crypto-monnaies/) (MonCoachData).

Exécute les cellules **dans l'ordre**. Le code est testé avec Python 3.13, Keras 3 (backend TensorFlow), pandas 3 et scikit-learn 1.9. Les cours journaliers BTC/EUR (du 1er janvier 2021 au 21 octobre 2026) sont chargés depuis le dépôt GitHub du blog : rien à télécharger. Les résultats peuvent différer légèrement selon la machine et la version de Keras.

**Ce notebook illustre une technique, pas une stratégie d'investissement.**

## Importations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.metrics import mean_absolute_error

print(keras.__version__, keras.backend.backend())

## Chargement des données

Cours journaliers du Bitcoin en euros (paire BTC/EUR de Binance) : `open`, `high`, `low`, `close` et `volume`.

In [ ]:
url = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/prediction-du-prix-des-crypto-monnaies/btc-eur-daily.csv"
hist = pd.read_csv(url, parse_dates=['date'], index_col='date')

target_col = 'close'
print(hist.shape)
print(hist.head())

In [ ]:
print(hist.index.min(), hist.index.max())
print(hist.isna().sum().sum(), 'valeurs manquantes')
print(hist.index.to_series().diff().value_counts())

## Découpage entraînement / test

Le découpage est chronologique : les 80 % de jours les plus anciens servent à l'entraînement, les 20 % les plus récents au test.

In [ ]:
def split_train_test(df, test_size=0.2):
    split_row = len(df) - int(test_size * len(df))
    train_data = df.iloc[:split_row]
    test_data = df.iloc[split_row:]
    return train_data, test_data

train, test = split_train_test(hist, test_size=0.2)
print(f"entraînement : {train.index.min():%d/%m/%Y} -> {train.index.max():%d/%m/%Y} ({len(train)} jours)")
print(f"test         : {test.index.min():%d/%m/%Y} -> {test.index.max():%d/%m/%Y} ({len(test)} jours)")

## Visualisation

In [ ]:
def line_plot(line1, line2, label1=None, label2=None, title='', lw=2):
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.plot(line1, label=label1, linewidth=lw)
    ax.plot(line2, label=label2, linewidth=lw)
    ax.set_ylabel('prix [EUR]', fontsize=14)
    ax.set_title(title, fontsize=16)
    ax.legend(loc='best', fontsize=16)

line_plot(train[target_col], test[target_col], 'entraînement', 'test', title='Bitcoin (BTC/EUR)', lw=2)
plt.show()

## Préparation des données

Chaque exemple est une fenêtre de 5 jours, normalisée par rapport à son premier jour. La cible est le prix de clôture du jour suivant, exprimé en variation relative.

In [ ]:
def normalise_zero_base(df):
    return df / df.iloc[0] - 1

def extract_window_data(df, window_len=5, zero_base=True):
    window_data = []
    for idx in range(len(df) - window_len):
        tmp = df.iloc[idx: idx + window_len].copy()
        if zero_base:
            tmp = normalise_zero_base(tmp)
        window_data.append(tmp.values)
    return np.array(window_data)

def prepare_data(df, target_col, window_len=5, zero_base=True, test_size=0.2):
    train_data, test_data = split_train_test(df, test_size=test_size)
    X_train = extract_window_data(train_data, window_len, zero_base)
    X_test = extract_window_data(test_data, window_len, zero_base)
    y_train = train_data[target_col][window_len:].values
    y_test = test_data[target_col][window_len:].values
    if zero_base:
        y_train = y_train / train_data[target_col][:-window_len].values - 1
        y_test = y_test / test_data[target_col][:-window_len].values - 1
    return train_data, test_data, X_train, X_test, y_train, y_test

## Construction du modèle

In [ ]:
def build_lstm_model(input_data, output_size, neurons=100, activ_func='linear',
                     dropout=0.2, loss='mse', optimizer='adam'):
    model = keras.Sequential([
        keras.Input(shape=(input_data.shape[1], input_data.shape[2])),
        layers.LSTM(neurons),
        layers.Dropout(dropout),
        layers.Dense(units=output_size, activation=activ_func),
    ])
    model.compile(loss=loss, optimizer=optimizer)
    return model

## Paramètres

In [ ]:
keras.utils.set_random_seed(42)

window_len = 5
test_size = 0.2
zero_base = True
lstm_neurons = 100
epochs = 20
batch_size = 32
loss = 'mse'
dropout = 0.2
optimizer = 'adam'

## Entraînement

In [ ]:
train, test, X_train, X_test, y_train, y_test = prepare_data(
    hist, target_col, window_len=window_len, zero_base=zero_base, test_size=test_size)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

model = build_lstm_model(X_train, output_size=1, neurons=lstm_neurons,
                         dropout=dropout, loss=loss, optimizer=optimizer)
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                    verbose=2, shuffle=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, epochs + 1), history.history['loss'], linewidth=2)
ax.set_xticks(range(1, epochs + 1, 2))
ax.set_xlabel('époque', fontsize=14)
ax.set_ylabel('erreur quadratique moyenne (MSE)', fontsize=14)
ax.set_title("Évolution de l'erreur pendant l'entraînement", fontsize=16)
plt.show()

## Évaluation

Erreur absolue moyenne (MAE) sur les variations relatives, puis en euros.

In [ ]:
targets = test[target_col][window_len:]
preds = model.predict(X_test, verbose=0).squeeze()
print(mean_absolute_error(y_test, preds))

In [ ]:
preds = test[target_col].values[:-window_len] * (preds + 1)
preds = pd.Series(index=targets.index, data=preds)
line_plot(targets, preds, 'réel', 'prédiction', title='Bitcoin (BTC/EUR) : prix réel et prédit sur la période de test', lw=3)
plt.show()

In [ ]:
print(f"MAE du LSTM : {mean_absolute_error(targets, preds):,.0f} EUR")

## Comparaison avec une référence naïve

La référence naïve prédit que le prix de demain sera celui d'aujourd'hui.

In [ ]:
veille = test[target_col].shift(1)[window_len:]   # prix de la veille
print(f"MAE de la référence naïve : {mean_absolute_error(targets, veille):,.0f} EUR")

In [ ]:
vrai_sens = np.sign(targets - veille)
pred_sens = np.sign(preds - veille)
print(f"Sens de variation correctement prédit : {(vrai_sens == pred_sens).mean():.1%}")

In [ ]:
fin = targets.index[-45:]   # les 45 derniers jours
fig, ax = plt.subplots(figsize=(13, 7))
ax.plot(targets[fin], label='réel', linewidth=3)
ax.plot(preds[fin], label='prédiction du LSTM', linewidth=3)
ax.plot(veille[fin], label='prix de la veille', linewidth=2, linestyle='--', color='gray')
ax.set_ylabel('prix [EUR]', fontsize=14)
ax.set_title('Zoom sur les 45 derniers jours', fontsize=16)
ax.legend(loc='best', fontsize=16)
plt.show()

## Le résultat dépend-il de la graine aléatoire ?

On relance l'entraînement avec cinq graines différentes.

In [ ]:
for seed in range(5):
    keras.utils.set_random_seed(seed)
    m = build_lstm_model(X_train, output_size=1, neurons=lstm_neurons, dropout=dropout)
    m.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0, shuffle=True)
    p = test[target_col].values[:-window_len] * (m.predict(X_test, verbose=0).squeeze() + 1)
    sens = (np.sign(targets - veille) == np.sign(p - veille)).mean()
    print(f"graine {seed} : MAE = {mean_absolute_error(targets, p):,.0f} EUR | sens correct : {sens:.1%}")

## Pour rafraîchir les données (facultatif)

L'API publique de Binance ne demande pas de clé. Les données de l'article sont figées dans le fichier `btc-eur-daily.csv` pour que les résultats restent reproductibles.

In [ ]:
import requests

def telecharger_btc_eur(debut='2021-01-01', fin='2026-10-22', symbole='BTCEUR'):
    url = 'https://data-api.binance.vision/api/v3/klines'
    start = int(pd.Timestamp(debut, tz='UTC').timestamp() * 1000)
    end = int(pd.Timestamp(fin, tz='UTC').timestamp() * 1000) - 1
    lignes = []
    while start < end:
        res = requests.get(url, params={'symbol': symbole, 'interval': '1d', 'limit': 1000,
                                        'startTime': start, 'endTime': end}, timeout=30)
        res.raise_for_status()
        lot = res.json()
        if not lot:
            break
        lignes += lot
        start = lot[-1][0] + 86_400_000   # on repart du jour suivant
    df = pd.DataFrame(lignes).iloc[:, :6]
    df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
    df['date'] = pd.to_datetime(df['date'], unit='ms')
    return df.set_index('date').astype(float)

# hist = telecharger_btc_eur()   # à décommenter pour obtenir des données à jour